# An Einstein Metric on $S^2\times S^3$

The historical Egison demo bearing the name “Riemann curvature tensor of
$S^2\times S^3$” uses a non-diagonal five-dimensional metric, not the
elementary block product of two round metrics.  Its goal is the Einstein
identity

$$
\operatorname{Ric}_{ij}=4g_{ij},\qquad \mathcal R=20.
$$

This notebook preserves that metric and makes its symbolic workload
explicit.  Small metric and connection components are suitable interactive
outputs; the full Einstein residual is defined for deliberate evaluation.


## Coordinates and abbreviations

In the coordinate order $x=(\phi,\theta,\psi,y,\alpha)$, introduce

$$
p=1-y,\quad u=a-y^2,\quad
v=a-3y^2+2y^3,\quad q=a-2y+y^2.
$$

These factors expose the repeated structure of the source metric and make
its coordinate domain restrictions ($p u v\ne0$) visible.


In [1]:
declare symbol φ, θ, ψ, y, α, a: MathValue

def x : Vector MathValue := [| φ, θ, ψ, y, α |]

def p : MathValue := `(1 - y)
def u : MathValue := `(a - y^2)
def v : MathValue := `(a - 3 * y^2 + 2 * y^3)
def q : MathValue := `(a - 2 * y + y^2)


In [2]:
(p, u, v, q)


$((-y + 1), (-y^{2} + a), (2 y^{3} - 3 y^{2} + a), (y^{2} - 2 y + a))$

## Local frame and induced metric

Put $f=q/(6u)$.  The metric has the orthonormal-frame decomposition

$$
\begin{split}
ds^2={}&\frac p6(d\theta^2+\sin^2\theta\,d\phi^2)
  +\frac{p}{2v}dy^2
  +\frac{v}{9u}(d\psi-\cos\theta\,d\phi)^2\\
  &+\frac{2u}{p}
    \bigl(d\alpha+f(d\psi-\cos\theta\,d\phi)\bigr)^2.
\end{split}
$$

Each row of `e_i_j` below is the coordinate tangent vector $e_i$
expressed in this orthonormal frame.  As in the round-sphere notebooks,
Egison constructs every metric component from tangent-vector inner
products, $g_{ij}=e_i\mathbin{\cdot}e_j$.  The coupled $(1,3,5)$ block is
produced by the two appearances of $d\psi-\cos\theta\,d\phi$.


In [3]:
def f : MathValue := q / (6 * u)

def e_i_j : Matrix MathValue :=
  [| [| sqrt (p / 6) * sin θ
       , 0
       , -(sqrt (v / u) / 3) * cos θ
       , 0
       , -sqrt (2 * u / p) * f * cos θ |]
   , [| 0, sqrt (p / 6), 0, 0, 0 |]
   , [| 0, 0, sqrt (v / u) / 3, 0, sqrt (2 * u / p) * f |]
   , [| 0, 0, 0, sqrt (p / (2 * v)), 0 |]
   , [| 0, 0, 0, 0, sqrt (2 * u / p) |]
   |]_i_j

def g_i_j : Matrix MathValue :=
  generateTensor (\[i, j] -> V.* e_i_# e_j_#) [5, 5]


In [4]:
g_2_2


$\frac{1}{6} (-y + 1)$

In [5]:
g_4_4


$\frac{1}{2} (-y + 1) (2 y^{3} - 3 y^{2} + a)^{-1}$

In [6]:
(g_1_3, g_3_1, g_1_5, g_5_1)


$(\frac{-1}{9} \cos(θ) (-y^{2} + a)^{-1} (2 y^{3} - 3 y^{2} + a) + \frac{-1}{18} \cos(θ) (-y + 1)^{-1} (-y^{2} + a)^{-1} (y^{2} - 2 y + a)^{2}, \frac{-1}{9} \cos(θ) (-y^{2} + a)^{-1} (2 y^{3} - 3 y^{2} + a) + \frac{-1}{18} \cos(θ) (-y + 1)^{-1} (-y^{2} + a)^{-1} (y^{2} - 2 y + a)^{2}, \frac{-1}{3} \cos(θ) (-y + 1)^{-1} (y^{2} - 2 y + a), \frac{-1}{3} \cos(θ) (-y + 1)^{-1} (y^{2} - 2 y + a))$

## Connection and curvature pipeline

The first-kind symbols require only derivatives of the displayed metric.
Raising an index introduces the inverse of the coupled block.  The Riemann
and Ricci definitions are nevertheless identical to the lower-dimensional
examples:

$$
R^i{}_{jkl}=\partial_k\Gamma^i{}_{jl}-\partial_l\Gamma^i{}_{jk}
  +\Gamma^m{}_{jl}\Gamma^i{}_{mk}-\Gamma^m{}_{jk}\Gamma^i{}_{ml},
\qquad
\operatorname{Ric}_{ij}=R^m{}_{imj}.
$$


In [7]:
def g~i~j : Matrix MathValue := M.inverse g_#_#

def Γ_i_j_k : Tensor MathValue :=
  (1 / 2) * (∂/∂ g_i_k x~j + ∂/∂ g_i_j x~k - ∂/∂ g_j_k x~i)

def Γ~i_j_k : Tensor MathValue := withSymbols [m]
  g~i~m . Γ_m_j_k

def R~i_j_k_l : Tensor MathValue := withSymbols [m]
  ∂/∂ Γ~i_j_l x~k - ∂/∂ Γ~i_j_k x~l
    + Γ~m_j_l . Γ~i_m_k - Γ~m_j_k . Γ~i_m_l

def Ric_i_j : Matrix MathValue := withSymbols [m]
  sum (contract R~m_i_m_j)

def einsteinResidual_i_j : Matrix MathValue := withSymbols [i, j]
  expandAll' (Ric_i_j -' 4 *' g_i_j)


In [8]:
Γ_2_2_4


$\frac{-1}{12}$

In [9]:
Γ_2_4_2


$\frac{-1}{12}$

## Interpretation and evaluation strategy

The two sampled first-kind symbols come from the simple $g_{22}=p/6$
entry and show the expected symmetry $\Gamma_{ijk}=\Gamma_{ikj}$.  The
complete test is `einsteinResidual_#_#`; mathematically it is the zero
matrix, so the scalar curvature is $5\cdot4=20$.

Expanding all 25 residual entries involves the inverse coupled block and
many rational simplifications.  It is intentionally not an automatic
output cell: evaluate individual entries such as
`einsteinResidual_2_2` when investigating the identity interactively.
